# 09 · Todo conectado y cierre

⏱ **10 min** · Slides de la Intro: 27 (todo conectado en un solo flujo) y 29 (ejercicio: diseña el flujo)

**Objetivo:** juntar los tres módulos en un solo recorrido y cerrar.

```
M4 · Unity Catalog: gobierno de todo el recorrido
API ─► RAW ─► BRONZE ─► SILVER ─► GOLD ─► SQL warehouse ─► analistas
       M5 · Job y pipeline declarativo        M6 · Databricks SQL
```

| Parte | Qué haces |
|---|---|
| 1 | Dos caminos, un mismo resultado |
| 2 | El linaje de punta a punta |
| 3 | Mini quiz |
| 4 | Mapa de lo aprendido |

**Requiere:** notebooks 05, 06 y 08.

In [0]:
%run ./00_Setup

## Parte 1 · Dos caminos, un mismo resultado

Corremos el ETL una vez más (para que lea los mismos archivos que el pipeline declarativo) y comparamos las dos tablas gold.

In [0]:
load_bronze()
build_silver()
build_gold()

In [0]:
columnas = ["airdate", "network_name", "episodes", "shows", "total_minutes"]
con_job      = spark.table(GOLD).select(columnas)
con_pipeline = spark.table(DLT_GOLD).select(columnas)

print("Filas en gold del Job     :", con_job.count())
print("Filas en gold del pipeline:", con_pipeline.count())
print("Filas distintas           :", con_job.exceptAll(con_pipeline).count() + con_pipeline.exceptAll(con_job).count(), "← debe ser 0")

👀 **Qué observar:** el Job (imperativo) y el pipeline (declarativo) llegan al mismo resultado. El archivo con errores no afectó a ninguno:
en uno lo quitó un `where`, en el otro una expectativa.

## Parte 2 · El linaje de punta a punta

**En la interfaz:** **Catalog → lab_tvmaze_m56 → resumen_diario → Lineage → See lineage graph**.
Verás todo el recorrido: `bronze_schedule → silver_episodes → gold_network_daily → resumen_diario`, con el Job que escribió las
primeras y la consulta del warehouse que creó la última.

Con SQL (las tablas del sistema llegan con retraso, hasta unos 20 minutos; si sale vacío, vuelve a ejecutar más tarde):

In [0]:
try_sql("""
    SELECT DISTINCT source_table_name AS origen, target_table_name AS destino, entity_type AS escrito_por
    FROM system.access.table_lineage
    WHERE target_table_schema = 'lab_tvmaze_m56' AND source_table_name IS NOT NULL
    ORDER BY destino
""")

In [0]:
%sql
-- Lo que puede ver el grupo de analistas en todo el esquema
SELECT table_name, grantee, privilege_type, inherited_from
FROM information_schema.table_privileges
WHERE table_schema = 'lab_tvmaze_m56' AND grantee = 'account users'

👀 **Qué observar:** el gobierno acompaña todo el recorrido: cada tabla tiene dueño, linaje y permisos, sin importar si la escribió
un notebook, un Job, un pipeline o una consulta del warehouse.

## Parte 3 · Mini quiz

1. **Unity Catalog:** ¿qué tres permisos necesita un analista para leer `workspace.lab_tvmaze_m56.gold_network_daily`?
2. **Unity Catalog:** ¿en qué se diferencian una *storage credential* y una *external location*?
3. **Workflows:** si la tarea `validacion` falla, ¿qué pasa con `publicacion` y con la tabla gold?
4. **Delta Live Tables:** ¿qué pasa con una fila que no cumple un `EXPECT` sin `ON VIOLATION`?
5. **Databricks SQL:** ¿qué es un SQL warehouse y qué pasa cuando nadie lo usa?
6. **Databricks SQL:** ¿por qué el analista y el ingeniero obtienen el mismo número?

*(Antes de seguir, piensa tu respuesta.)*

In [0]:
respuestas = {
    "1. Permisos": "USE CATALOG en workspace + USE SCHEMA en lab_tvmaze_m56 + SELECT en la tabla; se heredan hacia abajo (notebook 02).",
    "2. Almacenamiento": "La storage credential es la identidad autorizada a entrar al almacenamiento; la external location es una ruta + la credencial que la protege (notebook 03).",
    "3. Workflows": "publicacion no corre (UPSTREAM_FAILED) y gold queda como estaba; llega la alerta y se puede usar Repair run (notebook 05).",
    "4. Expectativas": "La fila entra igual y el fallo queda en las métricas; con DROP ROW se descarta y con FAIL UPDATE se detiene la actualización (notebook 06).",
    "5. SQL warehouse": "Es el cómputo dedicado a SQL de los analistas y las herramientas de BI; se enciende al recibir una consulta y se apaga solo tras unos minutos sin uso (notebook 07).",
    "6. Una fuente de verdad": "Porque los dos cómputos leen la misma tabla Delta gobernada por Unity Catalog: no hay copias entre el lago y el warehouse (notebooks 07 y 08).",
}
for pregunta, respuesta in respuestas.items():
    print(f"{pregunta}\n   → {respuesta}\n")

## Parte 4 · Mapa de lo aprendido

| Tema de la Intro | Slide | Notebook | Evidencia |
|---|---|---|---|
| Por qué gobernar: acceso, auditoría, linaje | 17 | 02, 03, 09 | `GRANT`, `system.query.history`, pestaña Lineage |
| La jerarquía de Unity Catalog | 18 | 01 | `CREATE CATALOG`, nombre de tres niveles, `information_schema` |
| Acceso seguro al almacenamiento | 19 | 03 | Tablas administradas, `SHOW EXTERNAL LOCATIONS` |
| El flujo ETL y la arquitectura medallion | 20 | 04 | `bronze_schedule` → `silver_episodes` → `gold_network_daily` |
| Workflows: orquestar tareas | 21 | 05 | Job de 4 tareas con parámetros, reintentos, horario y alerta |
| Delta Live Tables: declarativo | 22 | 06 | Pipeline SQL con `STREAMING TABLE` y expectativas |
| Databricks SQL: analítica | 23 | 07 | SQL warehouse, editor SQL, dashboard, alerta |
| Consultas y gestión de datos | 24 | 08 | Consultas, `CREATE`, `ALTER`, `GRANT`, `OPTIMIZE` en el warehouse |

El ejercicio del slide 29 (la cadena de tiendas con archivos CSV de 50 sucursales) ya tiene respuesta para tres preguntas:
**¿quién puede ver qué?** (módulo 4), **¿cómo lo automatizas?** (módulo 5) y **¿cómo lo consultan los analistas?** (módulo 6).

## Para seguir practicando

1. **Otra expectativa:** agrega al SQL del pipeline `CONSTRAINT temporada_valida EXPECT (season > 0)`, vuelve a ejecutar el notebook 06 y mira cuántas filas fallan.
2. **El pipeline dentro del Job:** agrega al Job una tarea `pipeline_task` (paso 7 del notebook 06) para que el Job dispare el pipeline.
3. **Un dashboard sobre el pipeline declarativo:** repite el dashboard del notebook 07 con `dlt_gold_network_daily` y comprueba que muestra lo mismo.

**Siguiente en el curso:** Módulo 7 · Asset Bundles (el Job y el pipeline de este laboratorio, desplegados como código).

## 🧹 Limpieza (opcional)
Elimina **todo** lo creado por el laboratorio. Quita los `#` para ejecutarla. El dashboard y la alerta se borran en la interfaz.

In [0]:
# delete_job_and_pipeline()                                  # el Job y el pipeline (con sus tablas dlt_)
# spark.sql(f"DROP SCHEMA `{CATALOG}`.{SCHEMA} CASCADE")     # tablas, vistas, funciones y el volumen